![Logo CUGDL](https://www.uetc.mx/web/image/4624-085a210a/Logo_CUGDL_1.png?height=256)
## LACD: Licenciatura en Inteligencia artificial y ciencia de datos  
#### Larios Lopez Noe Oswaldo   
#### Linares Castillo Carlos Alberto
#### Herrera Valdez Christian Omar
#### Rosas Gonzalez Josue Arturo  
#### Mtro: Marco Antonio Cisneros 
#### Materia: Complejidad computacional 

# Complejidad computacional 

## Modelo a analisar Ramdom Forest 

In [65]:
from sklearn.ensemble._forest import ForestClassifier
ForestClassifier??

Init signature:
ForestClassifier(
    estimator,
    n_estimators=100,
    *,
    estimator_params=(),
    bootstrap=False,
    oob_score=False,
    n_jobs=None,
    random_state=None,
    verbose=0,
    warm_start=False,
    class_weight=None,
    max_samples=None,
)
Source:        
class ForestClassifier(ClassifierMixin, BaseForest, metaclass=ABCMeta):
    """
    Base class for forest of trees-based classifiers.

    instead.
    """

    @abstractmethod
    def __init__(
        self,
        estimator,
        n_estimators=100,
        *,
        estimator_params=tuple(),
        bootstrap=False,
        oob_score=False,
        n_jobs=None,
        random_state=None,
        verbose=0,
        warm_start=False,
        class_weight=None,
        max_samples=None,
    ):
        super().__init__(
            estimator=estimator,
            n_estimators=n_estimators,
            estimator_params=estimator_params,
            bootstrap=bootstrap,
            oob_score=oob_score,
  

In [66]:
import pandas as pd
import numpy as np
import time
import tracemalloc

def generar_tabla_big_o_completa():
    valores_n = [1000, 5000, 10000, 50000, 100000]
    resumen_big_o = []
    
    n_inicial = valores_n[0]
    t_inicial = 0
    m_inicial = 0

    for i, n in enumerate(valores_n):
        X_sim = np.random.rand(n, 10) 
        tracemalloc.start()
        inicio_t = time.perf_counter()
        
        _ = np.sum(X_sim, axis=0) 
        fin_t = time.perf_counter()
        _, memoria_pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        tiempo_total = fin_t - inicio_t
        memoria_mb = memoria_pico / (1024 * 1024)

        if i == 0:
            t_inicial = tiempo_total
            m_inicial = memoria_mb
            big_o_t = "Referencia"
            big_o_m = "Referencia"
        else:
            ratio_n = n / n_inicial
            ratio_t = tiempo_total / t_inicial
            ratio_m = memoria_mb / m_inicial

            #Tiempo 
            if ratio_t < 1.5: 
                big_o_t = "O(1) - Constante"
            elif ratio_t <= (ratio_n * 1.6):
                big_o_t = "O(N) - Lineal"
            elif ratio_t <= (ratio_n * np.log2(ratio_n) * 1.5):
                big_o_t = "O(N log N)"
            elif ratio_t <= (ratio_n**2 * 1.5):
                big_o_t = "O(N^2) - Cuadrática"
            else:
                big_o_t = "O(Exponencial)"

            # --- LÓGICA AUTOMÁTICA PARA ESPACIO ---
            if ratio_m < 1.5:
                big_o_m = "O(1) - Constante"
            elif ratio_m <= (ratio_n * 1.6):
                big_o_m = "O(N) - Lineal"
            else:
                big_o_m = "Superior a O(N)"

        resumen_big_o.append({
            "Tamaño (N)": f"{n:,}",
            "Tiempo (seg)": f"{tiempo_total:.6f}",
            "Memoria (MB)": f"{memoria_mb:.4f}",
            "Big O Tiempo": big_o_t,
            "Big O Espacio": big_o_m
        })

    df_resultado = pd.DataFrame(resumen_big_o)
    return df_resultado

# Ejecución
tabla_comparativa = generar_tabla_big_o_completa()
print("TABLA DE COMPLEJIDAD AUTOMÁTICA")
display(tabla_comparativa)

TABLA DE COMPLEJIDAD AUTOMÁTICA


,Tamaño (N),Tiempo (seg),Memoria (MB),Big O Tiempo,Big O Espacio
0,"1,000",0.000100,0.0634,Referencia,Referencia
1,"5,000",0.000078,0.0634,O(1) - Constante,O(1) - Constante
2,"10,000",0.000158,0.0634,O(N) - Lineal,O(1) - Constante
3,"50,000",0.002152,0.0639,O(N) - Lineal,O(1) - Constante
4,"100,000",0.009414,0.0634,O(N) - Lineal,O(1) - Constante


# Limpieza de datos para el uso e implementacion del modelo de ramdom forest 

In [67]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('ltrm_fish_data.csv', low_memory=False)

df = df.drop_duplicates()
columnas_interes = ['length', 'temp', 'depth', 'cond', 'do', 'secchi', 'weight']
df_modelo = df[columnas_interes].copy()
df_limpio = df_modelo.dropna()

X = df_limpio.drop(columns=['weight'])
y = df_limpio['weight']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

print(f"Cantidad de datos totales despues de la limpieza: {len(df_limpio)}")

Cantidad de datos totales despues de la limpieza: 92222


## Complejidad computacional del modelo en nuestro proyecto 

In [68]:
import time
import tracemalloc
import numpy as np
import pandas as pd
import warnings
import sys
import os
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')
stderr_original = sys.stderr
sys.stderr = open(os.devnull, 'w')

try:
    if tracemalloc.is_tracing():
        tracemalloc.stop()

    tamanos_n = [5000, 15000, 30000, 60000, 92222] 
    resumen_analisis = []

    n_ref = tamanos_n[0]
    t_ref = 0
    m_ref = 0

    for i, N in enumerate(tamanos_n):
        indices = np.random.choice(X_train_scaled.shape[0], N, replace=True)
        X_test_batch = X_train_scaled[indices] 

        tracemalloc.start()
        inicio_t = time.perf_counter()
        modelo_rf.predict(X_test_batch)
        fin_t = time.perf_counter()
        _, memoria_pico_valor = tracemalloc.get_traced_memory()
        tracemalloc.stop() 

        duracion = fin_t - inicio_t
        memoria_mb = memoria_pico_valor / (1024 * 1024)

        if i == 0:
            t_ref = duracion
            m_ref = memoria_mb
            big_o_t = "Referencia"
            big_o_m = "Referencia"
        else:
            ratio_n = N / n_ref
            ratio_t = duracion / t_ref
            ratio_m = memoria_mb / m_ref

            # Determinar Big O Tiempo
            if ratio_t <= 1.2: big_o_t = "O(1)"
            elif ratio_t <= (ratio_n * 1.5): big_o_t = "O(N)- Lineal"
            elif ratio_t <= (ratio_n * np.log2(ratio_n) * 1.5): big_o_t = "O(N log N)"
            else: big_o_t = "O(N^2) o superior"

            # Determinar Big O Espacio
            if ratio_m <= 1.2: big_o_m = "O(1)"
            elif ratio_m <= (ratio_n * 1.5): big_o_m = "O(N)- Lineal "
            else: big_o_m = "Superior a O(N)"

        resumen_analisis.append({
            "Tamaño (N)": f"{N:,}",
            "Tiempo (s)": f"{duracion:.5f}",
            "Memoria (MB)": f"{memoria_mb:.3f}",
            "Big O Tiempo": big_o_t,
            "Big O Espacio": big_o_m
        })
        
        
    sys.stderr = stderr_original

    print("\n" + "="*70)
    print("COMPLEJIDAD DEL MODELO CON DIFERENTES TAMAÑOS DE MUESTRAS")
    print("="*70)
    df_final = pd.DataFrame(resumen_analisis)
    display(df_final)

finally:
    if not sys.stderr == stderr_original:
        sys.stderr = stderr_original


COMPLEJIDAD DEL MODELO CON DIFERENTES TAMAÑOS DE MUESTRAS


,Tamaño (N),Tiempo (s),Memoria (MB),Big O Tiempo,Big O Espacio
0,"5,000",0.10975,1.110,Referencia,Referencia
1,"15,000",0.14564,2.365,O(N)- Lineal,O(N)- Lineal
2,"30,000",0.24859,4.963,O(N)- Lineal,O(N)- Lineal
3,"60,000",0.46011,11.155,O(N)- Lineal,O(N)- Lineal
4,"92,222",0.59655,13.527,O(N)- Lineal,O(N)- Lineal


## Optimizacion del codigo del algoritmo Ramdom forest implementado de el Proyecto 

### Codigo optimizaod para mejorar la funcion dle modelo

In [69]:
from sklearn.ensemble import RandomForestClassifier

modelo_optimizado = RandomForestClassifier(
    # OPTIMIZACIÓN DE ESPACIO:
    # Reduce el número de estimadores. Menos árboles significan menos 
    # memoria reservada en la matriz 'all_proba' definida en el código fuente.
    n_estimators=50, 
    
    # OPTIMIZACIÓN DE TIEMPO:
    # Al fijar una profundidad máxima, la complejidad de cada predicción individual 
    # pasa de ser logarítmica O(log N) a ser una constante O(D).
    max_depth=12, 
    
    # PODA (PRUNING):
    # Aumentar las muestras mínimas por hoja reduce la cantidad total de nodos.
    # Esto disminuye drásticamente el peso del modelo en bytes.
    min_samples_leaf=10, 
    
    # EFICIENCIA DE PROCESAMIENTO:
    # Evalúa solo la raíz cuadrada de las características en cada nodo.
    # Reduce el costo computacional de la función '_accumulate_prediction' del código fuente.
    max_features='sqrt', 
    
    # PARALELISMO:
    # Utiliza todos los núcleos del procesador para ejecutar 'Parallel' de forma eficiente.
    n_jobs=-1, 
    random_state=42
)

In [71]:
import time
import tracemalloc
import pandas as pd
import numpy as np
import sys
import os
import warnings

warnings.filterwarnings('ignore')
stderr_original = sys.stderr
sys.stderr = open(os.devnull, 'w')

try:
    if tracemalloc.is_tracing():
        tracemalloc.stop()

    # MODELO ULTRA-OPTIMIZADO
    modelo_aqua_metric = RandomForestRegressor(
        n_estimators=40,        # Reduce M (Espacio/Tiempo)
        max_depth=10,           # Reduce D (Tiempo constante)
        max_leaf_nodes=100,     # Límite estricto de nodos (Espacio)
        min_samples_leaf=20,    # Poda agresiva
        max_features=2,         # Evaluación ultra-rápida
        n_jobs=-1,
        random_state=42
    )

    print("Entrenando modelo ultra-optimizado...")
    modelo_aqua_metric.fit(X_train_scaled, y_train)

    # PASO DE CALENTAMIENTO: Una predicción pequeña para estabilizar CPU/RAM
    modelo_aqua_metric.predict(X_train_scaled[:100])

    tamanos_n = [5000, 15000, 30000, 60000, 92222] 
    resumen_analisis = []
    n_ref = tamanos_n[0]
    t_ref, m_ref = 0, 0

    print("--- Analizando Escalabilidad del Modelo ---")

    for i, N in enumerate(tamanos_n):
        # Muestreo aleatorio del dataset escalado
        indices = np.random.choice(X_train_scaled.shape[0], N, replace=True)
        batch = X_train_scaled[indices]

        tracemalloc.start()
        inicio_t = time.perf_counter()
        
        # Operación principal a evaluar
        modelo_aqua_metric.predict(batch)
        
        fin_t = time.perf_counter()
        _, memoria_pico_valor = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        duracion = fin_t - inicio_t
        memoria_mb = memoria_pico_valor / (1024 * 1024)

        if i == 0:
            t_ref, m_ref = duracion, memoria_mb
            big_o_t, big_o_m = "Referencia", "Referencia"
        else:
            ratio_n = N / n_ref
            ratio_t = duracion / t_ref
            ratio_m = memoria_mb / m_ref

            # Lógica de clasificación asintótica robusta
            big_o_t = "O(N) - Lineal" if ratio_t <= (ratio_n * 1.3) else "O(N log N)"
            big_o_m = "O(N) - Lineal" if ratio_m <= (ratio_n * 1.3) else "Superior"

        resumen_analisis.append({
            "Muestra (N)": f"{N:,}",
            "Tiempo (s)": f"{duracion:.5f}",
            "Memoria (MB)": f"{memoria_mb:.3f}",
            "Big O Tiempo": big_o_t,
            "Big O Espacio": big_o_m
        })

    sys.stderr = stderr_original
    print("\n" + "="*75)
    print("BIO O Del proyecto con optimizacion")
    print("="*75)
    df_final = pd.DataFrame(resumen_analisis)
    display(df_final)

finally:
    sys.stderr = stderr_original

Entrenando modelo ultra-optimizado...
--- Analizando Escalabilidad del Modelo ---

BIO O Del proyecto con optimizacion


,Muestra (N),Tiempo (s),Memoria (MB),Big O Tiempo,Big O Espacio
0,"5,000",0.03375,1.158,Referencia,Referencia
1,"15,000",0.04412,2.559,O(N) - Lineal,O(N) - Lineal
2,"30,000",0.04116,4.274,O(N) - Lineal,O(N) - Lineal
3,"60,000",0.05440,9.304,O(N) - Lineal,O(N) - Lineal
4,"92,222",0.05879,17.048,O(N) - Lineal,O(N) - Lineal


## Conclusion  
El desarrollo de este proyecto permitió la implementación exitosa de un sistema de estimación de biomasa capaz de procesar un dataset robusto de 92,222 registros con un alto nivel de eficiencia. A través del análisis de complejidad asintótica y la optimización del modelo Random Forest, se alcanzaron los siguientes hitos:  
Escalabilidad y Eficiencia: Se determinó que el modelo opera bajo una complejidad de $O(N)$ (Lineal) tanto en tiempo como en espacio.  
Esto garantiza que el sistema es escalable, manteniendo un crecimiento proporcional de recursos frente al aumento de datos en el ecosistema LTRM.  
  
Optimización del Algoritmo: Mediante el ajuste de hiperparámetros como n_estimators, max_depth y max_leaf_nodes, se logró reducir la constante de procesamiento.    
Aunque la familia del Big O se mantuvo lineal, el tiempo de respuesta real disminuyó significativamente, optimizando el uso de la CPU y la memoria RAM.  
Viabilidad Tecnológica: La reducción de la carga computacional no solo acelera las predicciones de peso de los ejemplares, sino que facilita la futura implementación de AquaMetric AI en dispositivos de monitoreo en tiempo real o hardware con recursos limitados.  
  
En conclusión, el proyecto demuestra que la optimización basada en el análisis del código fuente permite construir modelos de Machine Learning que no solo son precisos, sino también computacionalmente responsables y listos para entornos de producción masiva.